### Modules nécessaires

In [115]:
using Pkg
Pkg.activate("Projet_env")
Pkg.add("BenchmarkTools")
Pkg.add("SuiteSparseMatrixCollection")
Pkg.add("MatrixMarket")

using LinearAlgebra, SparseArrays
using MatrixMarket
using SuiteSparseMatrixCollection
using Test
using BenchmarkTools

  Activating project at `~/Desktop/GitHub/Algebre Lineaire Numerique/MTH8211_Projet/Projet_env`
   Resolving package versions...
  No Changes to `~/Desktop/GitHub/Algebre Lineaire Numerique/MTH8211_Projet/Projet_env/Project.toml`
  No Changes to `~/Desktop/GitHub/Algebre Lineaire Numerique/MTH8211_Projet/Projet_env/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Desktop/GitHub/Algebre Lineaire Numerique/MTH8211_Projet/Projet_env/Project.toml`
  No Changes to `~/Desktop/GitHub/Algebre Lineaire Numerique/MTH8211_Projet/Projet_env/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Desktop/GitHub/Algebre Lineaire Numerique/MTH8211_Projet/Projet_env/Project.toml`
  No Changes to `~/Desktop/GitHub/Algebre Lineaire Numerique/MTH8211_Projet/Projet_env/Manifest.toml`


### Utilitaire

In [116]:
struct LBL{T}
    """
    Structure pour une matrice factorisée selon PAP' = LBL' (Bunch-Parlett et Bunch-Kaufman)
    """
    L::LowerTriangular{T, Matrix{T}}
    B::Hermitian{T, Tridiagonal{T, Vector{T}}}
    vec_P::Vector{Int}
end

function extract_L_and_B_from_LBL_inplace(A, vec_2by2)
    """
    Récupère la matrice L et B à partir de la forme compacte de la factorisation LBL'
        Entrée : - Matrice A sous la forme compacte de la factorisation LBL'
                 - Vecteur de position des blocs diagonaux 2x2 vec_2by2
        Sortie : - Matrice L triangulaire inférieure
                 - Matrice B tridiagonale hermitienne
    """
    n = size(A, 1)
    T = eltype(A)
    subdiagonal = zeros(T, n-1)
    diagonal = zeros(T, n)
    L = LowerTriangular(Matrix{T}(I, n, n))

    i_diagonal = 1
    while i_diagonal < n - 1
        if i_diagonal in vec_2by2
            # Matrice B
            subdiagonal[i_diagonal], subdiagonal[i_diagonal+1] = A[i_diagonal+1, i_diagonal], 0
            diagonal[i_diagonal], diagonal[i_diagonal + 1] = A[i_diagonal, i_diagonal], A[i_diagonal+1, i_diagonal+1]

            # Matrice L
            L[i_diagonal + 2:n, i_diagonal] .= A[i_diagonal + 2:n, i_diagonal]
            L[i_diagonal + 2:n, i_diagonal + 1] .= A[i_diagonal + 2:n, i_diagonal + 1]

            # Prochain bloc
            i_diagonal += 2
        else
            # Matrice B
            subdiagonal[i_diagonal] = 0
            diagonal[i_diagonal] = A[i_diagonal, i_diagonal]

            # Matrice L
            L[i_diagonal + 1:n, i_diagonal] .= A[i_diagonal + 1:n, i_diagonal]

            # Prochain bloc
            i_diagonal += 1
        end
    end

    if i_diagonal == n - 1
        subdiagonal[i_diagonal] = A[i_diagonal+1, i_diagonal]
        diagonal[i_diagonal], diagonal[i_diagonal+1]  = A[i_diagonal, i_diagonal], A[i_diagonal+1, i_diagonal+1]
    else
        diagonal[i_diagonal] = A[i_diagonal, i_diagonal]
    end

    B = Hermitian(Tridiagonal(subdiagonal, diagonal, conj(subdiagonal)))

    return L, B
end

function extract_P_from_vec_P(vec_P)
    """
    Retourne la matrice de permutation P correspondant à un vecteur de permutation vec_P
        Entrée : - Vecteur de permutation vec_P
        Sortie : - Matrice P
    """
    P = Matrix{Int}(I, n, n)
    P = P[vec_P, :]
    return P
end

function lmult_vec_P(M, vec_P)
    """
    Permute les lignes de la matrice M selon vec_P
        Entrée : - Matrice M
                 - Vecteur de permutation vec_P
        Sortie : - Matrice M permutée
    """
    return M[vec_P, :]
end

function rmult_vec_Pt(M, vec_P)
    """
    Permute les colonnes de la matrice M selon un vecteur de permutation vec_P
        Entrée : - Matrice M
                 - Vecteur de permutation vec_P
        Sortie : - Matrice M permutée
    """
    return M[:, vec_P]
end

function transpose_vec_P(vec_P)
    """
    Transpose un vecteur de permutation vec_P
        Entrée : - Vecteur de permutation vec_P
        Sortie : - Vecteur de permutation transposé vec_P_t
    """
    n = size(vec_P, 1)
    vec_P_t = zeros(Int, n)
    for i in 1:n
        vec_P_t[vec_P[i]] = i
    end
    return vec_P_t
end

function perm_r1_et_r2!(M, r1::Int, r2::Int)
    """
    Modifie une matrice M hermitienne en permutant les lignes et les colonnes d'indices r1 et r2, où r2 >= r1
        Entrée : - Matrice M hermitienne
                 - Indice r1
                 - Indice r2
    """
    if r2 < r1
        error("r2 doit être plus grand ou égal à r1")
    end

    n = size(M, 1)

    M[r1, r1], M[r2, r2] = M[r2, r2], M[r1, r1]
    M[r2, r1] = conj(M[r2, r1])
    for k in r1+1:r2-1
        M[k, r1], M[r2, k] = conj(M[r2, k]), conj(M[k, r1])
    end
    for i in r2+1:n
        M[i, r1], M[i, r2] = M[i, r2], M[i, r1]
    end
    for j in 1:r1-1
        M[r1, j], M[r2, j] = M[r2, j], M[r1, j]
    end
end

perm_r1_et_r2! (generic function with 1 method)

### Implémentation de Bunch-Parlett (Complete Pivoting)

### Implémentation de Bunch-Kaufman (Partial Pivoting)

In [118]:
function bunch_kaufman(A::LowerTriangular)
    """
    Factorise A selon la factorisation LBL' de Bunch-Kaufman, qui utilise la technique de pivotage partiel
        Entrée : - Matrice A carrée, hermitienne et indéfinie
        Sortie : - Structure qui contient :
                    - Matrice L triangulaire inférieure
                    - Matrice B tridiagonale hermitienne
                    - Vecteur de permutation vec_P
    """
    n = size(A, 1)
    T = eltype(A)
    subdiagonal = zeros(T, n-1)
    diagonal = zeros(T, n)
    L = LowerTriangular(Matrix{T}(I, n, n))
    vec_P = collect(1:n)

    alpha = (1 + sqrt(17))/8
    i_diagonal = 1
    while i_diagonal < n - 1
        ### Initialisation de la partie de A traitée
        A_ = view(A, i_diagonal:n, i_diagonal:n)
        n_ = size(A_, 1)

        ### Choix du pivot et pivotage
        w_1 = -1.0
        r = 1
        for i in 2:n_
            magnitude_a_i1 = abs(A_[i, 1])
            if magnitude_a_i1 > w_1
                w_1 = magnitude_a_i1
                r = i
            end
        end

        magnitude_a_11 = abs(A_[1, 1])
        if magnitude_a_11 >= alpha*w_1  # E de taille 1, P ne permute rien
            # Déterminant de E
            a_11_conj = conj(A_[1, 1])
            inv_det_E = a_11_conj/magnitude_a_11^2

            # Complément de Schur
            for i in 2:n_
                a_i1 = A_[i, 1]
                for j in 2:i
                    a_j1_conj = conj(A_[j, 1])
                    A_[i, j] -= inv_det_E * a_i1 * a_j1_conj
                end
            end
            
            # Calcul de B
            diagonal[i_diagonal] = A_[1, 1]
            subdiagonal[i_diagonal] = 0

            # Calcul de L
            for i in 2:n_
                L[i_diagonal+i-1, i_diagonal] = inv_det_E*A_[i, 1]
            end

            # Prochain bloc
            i_diagonal += 1
        else
            w_r = -1.0
            for j in 1:r-1
                magnitude_a_ir = abs(A_[r, j])
                if magnitude_a_ir > w_r
                    w_r = magnitude_a_ir
                end
            end
            for i in r+1:n_
                magnitude_a_ir = abs(A_[i, r])
                if magnitude_a_ir > w_r
                    w_r = magnitude_a_ir
                end
            end

            if magnitude_a_11*w_r >= alpha*w_1^2  # E de taille 1, P ne permute rien
                # Déterminant de E
                a_11_conj = conj(A_[1, 1])
                inv_det_E = a_11_conj/magnitude_a_11^2

                # Complément de Schur
                for i in 2:n_
                    a_i1 = A_[i, 1]
                    for j in 2:i
                        a_j1_conj = conj(A_[j, 1])
                        A_[i, j] -= inv_det_E * a_i1 * a_j1_conj
                    end
                end

                # Calcul de B
                diagonal[i_diagonal] = A_[1, 1]
                subdiagonal[i_diagonal] = 0

                # Calcul de L
                for i in 2:n_
                    L[i_diagonal+i-1, i_diagonal] = inv_det_E*A_[i, 1]
                end

                # Prochain bloc
                i_diagonal += 1
            else
                magnitude_a_rr = abs(A_[r, r])
                if magnitude_a_rr >= alpha*w_r  # E de taille 1, P permute 1 et r
                    # Permutation dans A_
                    perm_r1_et_r2!(A_, 1, r)

                    # Permutation dans L
                    for j in 1:i_diagonal-1
                        L[i_diagonal, j], L[i_diagonal+r-1, j] = L[i_diagonal+r-1, j], L[i_diagonal, j]
                    end
                    
                    # Permutation dans P
                    vec_P[i_diagonal], vec_P[i_diagonal+r-1] = vec_P[i_diagonal+r-1], vec_P[i_diagonal]

                    # Déterminant de E
                    a_11_conj = conj(A_[1, 1])
                    inv_det_E = a_11_conj/magnitude_a_rr^2

                    # Complément de Schur
                    for i in 2:n_
                        a_i1 = A_[i, 1]
                        for j in 2:i
                            a_j1_conj = conj(A_[j, 1])
                            A_[i, j] -= inv_det_E * a_i1 * a_j1_conj
                        end
                    end

                    # Calcul de B
                    diagonal[i_diagonal] = A_[1, 1]
                    subdiagonal[i_diagonal] = 0

                    # Calcul de L
                    for i in 2:n_
                        L[i_diagonal+i-1, i_diagonal] = inv_det_E*A_[i, 1]
                    end
                    
                    # Prochain bloc
                    i_diagonal += 1
                else  # E de taille 2, P permute 2 et r                    
                    # Permutation dans A_
                    perm_r1_et_r2!(A_, 2, r)

                    # Permutation dans L
                    for j in 1:i_diagonal-1
                        L[i_diagonal+1, j], L[i_diagonal+r-1, j] = L[i_diagonal+r-1, j], L[i_diagonal+1, j]
                    end
                    
                    # Permutation dans P
                    vec_P[i_diagonal+1], vec_P[i_diagonal+r-1] = vec_P[i_diagonal+r-1], vec_P[i_diagonal+1]

                    # Déterminant de E
                    e_11, e_22, e_21 = A_[1, 1], A_[2, 2], A_[2, 1]
                    e_12 = conj(e_21)
                    det_E = (e_11*e_22 - e_21*e_12)
                    det_E_conj = conj(det_E)
                    magnitude_det_E = abs(det_E)
                    inv_det_E = det_E_conj/magnitude_det_E^2

                    # Complément de Schur
                    for i in 3:n_
                        a_i1, a_i2 = A_[i, 1], A_[i, 2]
                        for j in 3:i
                            a_j1_conj, a_j2_conj = conj(A_[j, 1]), conj(A_[j, 2])
                            A_[i, j] -= inv_det_E*((a_i1*e_22 - a_i2*e_21)*a_j1_conj + (a_i2*e_11 - a_i1*e_12)*a_j2_conj)
                        end
                    end
                    
                    # Calcul de B
                    subdiagonal[i_diagonal], subdiagonal[i_diagonal+1] = A_[2, 1], 0
                    diagonal[i_diagonal], diagonal[i_diagonal + 1] = A_[1, 1], A_[2, 2]

                    # Calcul de L
                    for i in i_diagonal + 2:n
                        a_i1, a_i2 = A[i, i_diagonal], A[i, i_diagonal+1]
                        L[i, i_diagonal] = inv_det_E*(a_i1*e_22 - a_i2*e_21)
                        L[i, i_diagonal+1] = inv_det_E*(a_i2*e_11 - a_i1*e_12)
                    end

                    # Prochain bloc
                    i_diagonal += 2
                end
            end
        end
    end

    # Dernier bloc
    if i_diagonal == n - 1
        subdiagonal[i_diagonal] = A[i_diagonal+1, i_diagonal]
        diagonal[i_diagonal], diagonal[i_diagonal+1]  = A[i_diagonal, i_diagonal], A[i_diagonal+1, i_diagonal+1]
    else
        diagonal[i_diagonal] = A[i_diagonal, i_diagonal]
    end

    B = Hermitian(Tridiagonal(subdiagonal, diagonal, conj(subdiagonal)))
    result = LBL(L, B, vec_P)
    return result
end

function bunch_kaufman!(A::LowerTriangular)
    """
    Modifie A sous la forme compacte de la factorisation LBL' de Bunch-Kaufman, qui utilise la technique du pivotage partiel.
    L et B sont stockés en écrasant A, où L se situe en dessous des éléments blocs diagonaux de B.
    Entrée : - Matrice A carrée, hermitienne et indéfinie
    Sortie : - Vecteur de permutation vec_P
             - Vecteur de position des blocs diagonaux 2x2 vec_2by2
    """
    n = size(A, 1)
    vec_P = collect(1:n)
    vec_2by2 = Int[]

    alpha = (1 + sqrt(17))/8
    i_diagonal = 1
    while i_diagonal < n - 1
        ### Initialisation de la partie de A traitée
        A_ = view(A, i_diagonal:n, i_diagonal:n)
        n_ = size(A_, 1)

        ### Choix du pivot et pivotage
        w_1 = -1.0
        r = 1
        for i in 2:n_
            magnitude_a_i1 = abs(A_[i, 1])
            if magnitude_a_i1 > w_1
                w_1 = magnitude_a_i1
                r = i
            end
        end

        magnitude_a_11 = abs(A_[1, 1])
        if magnitude_a_11 >= alpha*w_1  # E de taille 1, P ne permute rien
            # Déterminant de E
            a_11_conj = conj(A_[1, 1])
            inv_det_E = a_11_conj/magnitude_a_11^2

            # Complément de Schur
            for i in 2:n_
                a_i1 = A_[i, 1]
                for j in 2:i
                    a_j1_conj = conj(A_[j, 1])
                    A_[i, j] -= inv_det_E * a_i1 * a_j1_conj
                end
            end
            
            # Calcul de L
            for i in 2:n_
                A_[i, 1] *= inv_det_E
            end

            # Prochain bloc
            i_diagonal += 1
        else
            w_r = -1.0
            for j in 1:r-1
                magnitude_a_ir = abs(A_[r, j])
                if magnitude_a_ir > w_r
                    w_r = magnitude_a_ir
                end
            end
            for i in r+1:n_
                magnitude_a_ir = abs(A_[i, r])
                if magnitude_a_ir > w_r
                    w_r = magnitude_a_ir
                end
            end

            if magnitude_a_11*w_r >= alpha*w_1^2  # E de taille 1, P ne permute rien
                # Déterminant de E
                a_11_conj = conj(A_[1, 1])
                inv_det_E = a_11_conj/magnitude_a_11^2

                # Complément de Schur
                for i in 2:n_
                    a_i1 = A_[i, 1]
                    for j in 2:i
                        a_j1_conj = conj(A_[j, 1])
                        A_[i, j] -= inv_det_E * a_i1 * a_j1_conj
                    end
                end

                # Calcul de L
                for i in 2:n_
                    A_[i, 1] *= inv_det_E
                end

                # Prochain bloc
                i_diagonal += 1
            else
                magnitude_a_rr = abs(A_[r, r])
                if magnitude_a_rr >= alpha*w_r  # E de taille 1, P permute 1 et r
                    # Permutation dans A_
                    perm_r1_et_r2!(A_, 1, r)

                    # Permutation dans L
                    for j in 1:i_diagonal-1
                        A[i_diagonal, j], A[i_diagonal+r-1, j] = A[i_diagonal+r-1, j], A[i_diagonal, j]
                    end
                    
                    # Permutation dans P
                    vec_P[i_diagonal], vec_P[i_diagonal+r-1] = vec_P[i_diagonal+r-1], vec_P[i_diagonal]

                    # Déterminant de E
                    a_11_conj = conj(A_[1, 1])
                    inv_det_E = a_11_conj/magnitude_a_rr^2

                    # Complément de Schur
                    for i in 2:n_
                        a_i1 = A_[i, 1]
                        for j in 2:i
                            a_j1_conj = conj(A_[j, 1])
                            A_[i, j] -= inv_det_E * a_i1 * a_j1_conj
                        end
                    end

                    # Calcul de L
                    for i in 2:n_
                        A_[i, 1] *= inv_det_E
                    end
                    
                    # Prochain bloc
                    i_diagonal += 1
                else  # E de taille 2, P permute 2 et r
                    push!(vec_2by2, i_diagonal)
                    
                    # Permutation dans A_
                    perm_r1_et_r2!(A_, 2, r)

                    # Permutation dans L
                    for j in 1:i_diagonal-1
                        A[i_diagonal+1, j], A[i_diagonal+r-1, j] = A[i_diagonal+r-1, j], A[i_diagonal+1, j]
                    end
                    
                    # Permutation dans P
                    vec_P[i_diagonal+1], vec_P[i_diagonal+r-1] = vec_P[i_diagonal+r-1], vec_P[i_diagonal+1]

                    # Déterminant de E
                    e_11, e_22, e_21 = A_[1, 1], A_[2, 2], A_[2, 1]
                    e_12 = conj(e_21)
                    det_E = (e_11*e_22 - e_21*e_12)
                    det_E_conj = conj(det_E)
                    magnitude_det_E = abs(det_E)
                    inv_det_E = det_E_conj/magnitude_det_E^2

                    # Complément de Schur
                    for i in 3:n_
                        a_i1, a_i2 = A_[i, 1], A_[i, 2]
                        for j in 3:i
                            a_j1_conj, a_j2_conj = conj(A_[j, 1]), conj(A_[j, 2])
                            A_[i, j] -= inv_det_E*((a_i1*e_22 - a_i2*e_21)*a_j1_conj + (a_i2*e_11 - a_i1*e_12)*a_j2_conj)
                        end
                    end

                    # Calcul de L
                    for i in 3:n_
                        a_i1, a_i2 = A_[i, 1], A_[i, 2]
                        A_[i, 1] = inv_det_E*(a_i1*e_22 - a_i2*e_21)
                        A_[i, 2] = inv_det_E*(a_i2*e_11 - a_i1*e_12)
                    end

                    # Prochain bloc
                    i_diagonal += 2
                end
            end
        end
    end

    return vec_P, vec_2by2
end

bunch_kaufman! (generic function with 1 method)

### Implémentation de Bunch-Kaufman (Rook Pivoting)

In [119]:
function bunch_kaufman_rook(A::LowerTriangular)
    """
    Factorise A selon la factorisation LBL' de Bunch-Kaufman, qui utilise la technique de pivotage partiel
        Entrée : - Matrice A carrée, hermitienne et indéfinie
        Sortie : - Structure qui contient :
                    - Matrice L triangulaire inférieure
                    - Matrice B tridiagonale hermitienne
                    - Vecteur de permutation vec_P
    """
    n = size(A, 1)
    T = eltype(A)
    subdiagonal = zeros(T, n-1)
    diagonal = zeros(T, n)
    L = LowerTriangular(Matrix{T}(I, n, n))
    vec_P = collect(1:n)

    alpha = (1 + sqrt(17))/8
    i_diagonal = 1
    while i_diagonal < n - 1
        ### Initialisation de la partie de A traitée
        A_ = view(A, i_diagonal:n, i_diagonal:n)
        n_ = size(A_, 1)

        ### Choix du pivot et pivotage
        w_1 = -1.0
        r = 1
        for i in 2:n_
            magnitude_a_i1 = abs(A_[i, 1])
            if magnitude_a_i1 > w_1
                w_1 = magnitude_a_i1
                r = i
            end
        end

        magnitude_a_11 = abs(A_[1, 1])
        if magnitude_a_11 >= alpha*w_1  # E de taille 1, P ne permute rien
            # Déterminant de E
            a_11_conj = conj(A_[1, 1])
            inv_det_E = a_11_conj/magnitude_a_11^2

            # Complément de Schur
            for i in 2:n_
                a_i1 = A_[i, 1]
                for j in 2:i
                    a_j1_conj = conj(A_[j, 1])
                    A_[i, j] -= inv_det_E * a_i1 * a_j1_conj
                end
            end
            
            # Calcul de B
            diagonal[i_diagonal] = A_[1, 1]
            subdiagonal[i_diagonal] = 0

            # Calcul de L
            for i in 2:n_
                L[i_diagonal+i-1, i_diagonal] = inv_det_E*A_[i, 1]
            end

            # Prochain bloc
            i_diagonal += 1
        else
            index = 1
            w_index = w_1
            r_new = r
            repeat = true 
            while repeat
                w_r = -1.0
                # Subdiagonal
                for i in r+1:n_
                    magnitude_a_ir = abs(A_[i, r])
                    if magnitude_a_ir > w_r
                        w_r = magnitude_a_ir
                        r_new = i
                    end
                end
                # Superdiagonal
                for j in 1:r-1
                    magnitude_a_ir = abs(A_[r, j])
                    if magnitude_a_ir > w_r
                        w_r = magnitude_a_ir
                    end
                end

                magnitude_a_rr = abs(A_[r, r])
                if magnitude_a_rr >= alpha*w_r  # E de taille 1, P permute 1 et r
                    repeat = false
                    
                    # Permutation dans A_
                    perm_r1_et_r2!(A_, 1, r)

                    # Permutation dans L
                    for j in 1:i_diagonal-1
                        L[i_diagonal, j], L[i_diagonal+r-1, j] = L[i_diagonal+r-1, j], L[i_diagonal, j]
                    end
                    
                    # Permutation dans P
                    vec_P[i_diagonal], vec_P[i_diagonal+r-1] = vec_P[i_diagonal+r-1], vec_P[i_diagonal]

                    # Déterminant de E
                    a_11_conj = conj(A_[1, 1])
                    inv_det_E = a_11_conj/magnitude_a_rr^2

                    # Complément de Schur
                    for i in 2:n_
                        a_i1 = A_[i, 1]
                        for j in 2:i
                            a_j1_conj = conj(A_[j, 1])
                            A_[i, j] -= inv_det_E * a_i1 * a_j1_conj
                        end
                    end

                    # Calcul de B
                    diagonal[i_diagonal] = A_[1, 1]
                    subdiagonal[i_diagonal] = 0

                    # Calcul de L
                    for i in 2:n_
                        L[i_diagonal+i-1, i_diagonal] = inv_det_E*A_[i, 1]
                    end
                    
                    # Prochain bloc
                    i_diagonal += 1
                elseif w_index == w_r  # E de taille 2, P permute 1 et index, puis 2 et r     
                    repeat = false

                    # Permutation dans A_
                    perm_r1_et_r2!(A_, 1, index)
                    perm_r1_et_r2!(A_, 2, r)

                    # Permutation dans L
                    for j in 1:i_diagonal-1
                        L[i_diagonal, j], L[i_diagonal+index-1, j] = L[i_diagonal+index-1, j], L[i_diagonal, j]
                    end
                    for j in 1:i_diagonal-1
                        L[i_diagonal+1, j], L[i_diagonal+r-1, j] = L[i_diagonal+r-1, j], L[i_diagonal+1, j]
                    end
                    
                    # Permutation dans P
                    vec_P[i_diagonal], vec_P[i_diagonal+index-1] = vec_P[i_diagonal+index-1], vec_P[i_diagonal]
                    vec_P[i_diagonal+1], vec_P[i_diagonal+r-1] = vec_P[i_diagonal+r-1], vec_P[i_diagonal+1]

                    # Déterminant de E
                    e_11, e_22, e_21 = A_[1, 1], A_[2, 2], A_[2, 1]
                    e_12 = conj(e_21)
                    det_E = (e_11*e_22 - e_21*e_12)
                    det_E_conj = conj(det_E)
                    magnitude_det_E = abs(det_E)
                    inv_det_E = det_E_conj/magnitude_det_E^2

                    # Complément de Schur
                    for i in 3:n_
                        a_i1, a_i2 = A_[i, 1], A_[i, 2]
                        for j in 3:i
                            a_j1_conj, a_j2_conj = conj(A_[j, 1]), conj(A_[j, 2])
                            A_[i, j] -= inv_det_E*((a_i1*e_22 - a_i2*e_21)*a_j1_conj + (a_i2*e_11 - a_i1*e_12)*a_j2_conj)
                        end
                    end
                    
                    # Calcul de B
                    subdiagonal[i_diagonal], subdiagonal[i_diagonal+1] = A_[2, 1], 0
                    diagonal[i_diagonal], diagonal[i_diagonal + 1] = A_[1, 1], A_[2, 2]

                    # Calcul de L
                    for i in i_diagonal + 2:n
                        a_i1, a_i2 = A[i, i_diagonal], A[i, i_diagonal+1]
                        L[i, i_diagonal] = inv_det_E*(a_i1*e_22 - a_i2*e_21)
                        L[i, i_diagonal+1] = inv_det_E*(a_i2*e_11 - a_i1*e_12)
                    end

                    # Prochain bloc
                    i_diagonal += 2
                else
                    index = r
                    w_index = w_r
                    r = r_new
                end
            end
        end
    end

    # Dernier bloc
    if i_diagonal == n - 1
        subdiagonal[i_diagonal] = A[i_diagonal+1, i_diagonal]
        diagonal[i_diagonal], diagonal[i_diagonal+1]  = A[i_diagonal, i_diagonal], A[i_diagonal+1, i_diagonal+1]
    else
        diagonal[i_diagonal] = A[i_diagonal, i_diagonal]
    end

    B = Hermitian(Tridiagonal(subdiagonal, diagonal, conj(subdiagonal)))
    result = LBL(L, B, vec_P)
    return result
end

function bunch_kaufman_rook!(A::LowerTriangular)
    """
    Modifie A sous la forme compacte de la factorisation LBL' de Bunch-Kaufman, qui utilise la technique du 'rook pivoting'.
    L et B sont stockés en écrasant A, où L se situe en dessous des éléments blocs diagonaux de B.
    Entrée : - Matrice A carrée, hermitienne et indéfinie
    Sortie : - Vecteur de permutation vec_P
             - Vecteur de position des blocs diagonaux 2x2 vec_2by2
    """
    n = size(A, 1)
    vec_P = collect(1:n)
    vec_2by2 = Int[]

    alpha = (1 + sqrt(17))/8
    i_diagonal = 1
    while i_diagonal < n - 1
        ### Initialisation de la partie de A traitée
        A_ = view(A, i_diagonal:n, i_diagonal:n)
        n_ = size(A_, 1)

        ### Choix du pivot et pivotage
        w_1 = -1.0
        r = 1
        for i in 2:n_
            magnitude_a_i1 = abs(A_[i, 1])
            if magnitude_a_i1 > w_1
                w_1 = magnitude_a_i1
                r = i
            end
        end

        magnitude_a_11 = abs(A_[1, 1])
        if magnitude_a_11 >= alpha*w_1  # E de taille 1, P ne permute rien
            # Déterminant de E
            a_11_conj = conj(A_[1, 1])
            inv_det_E = a_11_conj/magnitude_a_11^2

            # Complément de Schur
            for i in 2:n_
                a_i1 = A_[i, 1]
                for j in 2:i
                    a_j1_conj = conj(A_[j, 1])
                    A_[i, j] -= inv_det_E * a_i1 * a_j1_conj
                end
            end
            
            # Calcul de L
            for i in 2:n_
                A_[i, 1] *= inv_det_E
            end

            # Prochain bloc
            i_diagonal += 1
        else
            index = 1
            w_index = w_1
            r_new = r
            repeat = true 
            while repeat
                w_r = -1.0
                # Subdiagonal
                for i in r+1:n_
                    magnitude_a_ir = abs(A_[i, r])
                    if magnitude_a_ir > w_r
                        w_r = magnitude_a_ir
                        r_new = i
                    end
                end
                # Superdiagonal
                for j in 1:r-1
                    magnitude_a_ir = abs(A_[r, j])
                    if magnitude_a_ir > w_r
                        w_r = magnitude_a_ir
                    end
                end

                magnitude_a_rr = abs(A_[r, r])
                if magnitude_a_rr >= alpha*w_r  # E de taille 1, P permute 1 et r
                    repeat = false

                    # Permutation dans A_
                    perm_r1_et_r2!(A_, 1, r)

                    # Permutation dans L
                    for j in 1:i_diagonal-1
                        A[i_diagonal, j], A[i_diagonal+r-1, j] = A[i_diagonal+r-1, j], A[i_diagonal, j]
                    end
                    
                    # Permutation dans P
                    vec_P[i_diagonal], vec_P[i_diagonal+r-1] = vec_P[i_diagonal+r-1], vec_P[i_diagonal]

                    # Déterminant de E
                    a_11_conj = conj(A_[1, 1])
                    inv_det_E = a_11_conj/magnitude_a_rr^2

                    # Complément de Schur
                    for i in 2:n_
                        a_i1 = A_[i, 1]
                        for j in 2:i
                            a_j1_conj = conj(A_[j, 1])
                            A_[i, j] -= inv_det_E * a_i1 * a_j1_conj
                        end
                    end

                    # Calcul de L
                    for i in 2:n_
                        A_[i, 1] *= inv_det_E
                    end
                    
                    # Prochain bloc
                    i_diagonal += 1
                elseif w_index == w_r  # E de taille 2, P permute 1 et index, puis 2 et r
                    repeat = false

                    push!(vec_2by2, i_diagonal)
                    
                    # Permutation dans A_
                    perm_r1_et_r2!(A_, 1, index)
                    perm_r1_et_r2!(A_, 2, r)

                    # Permutation dans L
                    for j in 1:i_diagonal-1
                        A[i_diagonal, j], A[i_diagonal+index-1, j] = A[i_diagonal+index-1, j], A[i_diagonal, j]
                    end
                    for j in 1:i_diagonal-1
                        A[i_diagonal+1, j], A[i_diagonal+r-1, j] = A[i_diagonal+r-1, j], A[i_diagonal+1, j]
                    end
                    
                    # Permutation dans P
                    vec_P[i_diagonal], vec_P[i_diagonal+index-1] = vec_P[i_diagonal+index-1], vec_P[i_diagonal]
                    vec_P[i_diagonal+1], vec_P[i_diagonal+r-1] = vec_P[i_diagonal+r-1], vec_P[i_diagonal+1]

                    # Déterminant de E
                    e_11, e_22, e_21 = A_[1, 1], A_[2, 2], A_[2, 1]
                    e_12 = conj(e_21)
                    det_E = (e_11*e_22 - e_21*e_12)
                    det_E_conj = conj(det_E)
                    magnitude_det_E = abs(det_E)
                    inv_det_E = det_E_conj/magnitude_det_E^2

                    # Complément de Schur
                    for i in 3:n_
                        a_i1, a_i2 = A_[i, 1], A_[i, 2]
                        for j in 3:i
                            a_j1_conj, a_j2_conj = conj(A_[j, 1]), conj(A_[j, 2])
                            A_[i, j] -= inv_det_E*((a_i1*e_22 - a_i2*e_21)*a_j1_conj + (a_i2*e_11 - a_i1*e_12)*a_j2_conj)
                        end
                    end

                    # Calcul de L
                    for i in 3:n_
                        a_i1, a_i2 = A_[i, 1], A_[i, 2]
                        A_[i, 1] = inv_det_E*(a_i1*e_22 - a_i2*e_21)
                        A_[i, 2] = inv_det_E*(a_i2*e_11 - a_i1*e_12)
                    end

                    # Prochain bloc
                    i_diagonal += 2
                else
                    index = r
                    w_index = w_r
                    r = r_new
                end
            end
        end
    end

    return vec_P, vec_2by2
end

bunch_kaufman_rook! (generic function with 1 method)

### Validation avec différents types de données

#### Bunch-Kaufman (Partial Pivoting)

In [120]:
N = 25
m, n = 50, 25
for i in 1:N
    A = randn(m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    result = bunch_kaufman(K)
    L, B, vec_P = result.L, result.B, result.vec_P
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(L) == T && eltype(B) == T
end

for i in 1:N
    A = randn(ComplexF64, m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    result = bunch_kaufman(K)
    L, B, vec_P = result.L, result.B, result.vec_P
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(L) == T && eltype(B) == T
end

for i in 1:N
    A = randn(BigFloat, m, n) + im * randn(BigFloat, m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    result = bunch_kaufman(K)
    L, B, vec_P = result.L, result.B, result.vec_P
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(L) == T && eltype(B) == T
end

#### Bunch-Kaufman in-place (Partial Pivoting)

In [121]:
N = 25
m, n = 50, 25
for i in 1:N
    A = randn(m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    vec_P, vec_2by2 = bunch_kaufman!(K)
    L, B = extract_L_and_B_from_LBL_inplace(K, vec_2by2)
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(K) == T && eltype(L) == T && eltype(B) == T
end

for i in 1:N
    A = randn(ComplexF64, m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    vec_P, vec_2by2 = bunch_kaufman!(K)
    L, B = extract_L_and_B_from_LBL_inplace(K, vec_2by2)
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(K) == T && eltype(L) == T && eltype(B) == T
end

for i in 1:N
    A = randn(BigFloat, m, n) + im * randn(BigFloat, m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    vec_P, vec_2by2 = bunch_kaufman!(K)
    L, B = extract_L_and_B_from_LBL_inplace(K, vec_2by2)
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(K) == T && eltype(L) == T && eltype(B) == T
end

#### Bunch-Kaufman (Rook Pivoting)

In [122]:
N = 25
m, n = 50, 25
for i in 1:N
    A = randn(m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    result = bunch_kaufman_rook(K)
    L, B, vec_P = result.L, result.B, result.vec_P
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(L) == T && eltype(B) == T
end

for i in 1:N
    A = randn(ComplexF64, m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    result = bunch_kaufman_rook(K)
    L, B, vec_P = result.L, result.B, result.vec_P
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(L) == T && eltype(B) == T
end

for i in 1:N
    A = randn(BigFloat, m, n) + im * randn(BigFloat, m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    result = bunch_kaufman_rook(K)
    L, B, vec_P = result.L, result.B, result.vec_P
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(L) == T && eltype(B) == T
end

#### Bunch-Kaufman in-place (Rook Pivoting)

In [123]:
N = 25
m, n = 50, 25
for i in 1:N
    A = randn(m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    vec_P, vec_2by2 = bunch_kaufman_rook!(K)
    L, B = extract_L_and_B_from_LBL_inplace(K, vec_2by2)
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(K) == T && eltype(L) == T && eltype(B) == T
end

for i in 1:N
    A = randn(ComplexF64, m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    vec_P, vec_2by2 = bunch_kaufman_rook!(K)
    L, B = extract_L_and_B_from_LBL_inplace(K, vec_2by2)
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(K) == T && eltype(L) == T && eltype(B) == T
end

for i in 1:N
    A = randn(BigFloat, m, n) + im * randn(BigFloat, m, n)
    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K = copy(K_0)

    vec_P, vec_2by2 = bunch_kaufman_rook!(K)
    L, B = extract_L_and_B_from_LBL_inplace(K, vec_2by2)
    vec_Pt = transpose_vec_P(vec_P)
    K = L*B*L'
    K = lmult_vec_P(K, vec_Pt)
    K = rmult_vec_Pt(K, vec_Pt)
    K = LowerTriangular(K)
    @test norm(K - K_0) < 1e-10
    @test eltype(K) == T && eltype(L) == T && eltype(B) == T
end

### Benchmarking

Il semble y avoir un décalage entre la documentation de LinearAlgebra et la réalité : le paramètre 'rook' n'est pas reconnue dans la signature de bunchkaufman ou bunchkaufman!.
Elle semble ne pas être implémentée ou alors ne pas être accessible dans la version de Julia utilisée ici.

In [124]:
m, n = 50, 25
A = randn(m, n)
T = eltype(A)
K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
K_0_Hermitian = Hermitian([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])

bench_bunch_kaufman = @benchmark bunch_kaufman(K) setup=(K=copy($K_0))
bench_bunch_kaufman_inplace = @benchmark bunch_kaufman!(K) setup=(K=copy($K_0))
bench_bunch_kaufman_rook = @benchmark bunch_kaufman_rook(K) setup=(K=copy($K_0))
bench_bunch_kaufman_rook_inplace = @benchmark bunch_kaufman_rook!(K) setup=(K=copy($K_0))

bench_bunch_kaufman_LAPACK = @benchmark bunchkaufman(K) setup=(K=copy($K_0_Hermitian))
bench_bunch_kaufman_inplace_LAPACK = @benchmark bunchkaufman!(K) setup=(K=copy($K_0_Hermitian))
# bench_bunch_kaufman_rook_LAPACK = @benchmark bunchkaufman(K, rook=true) setup=(K=copy($K_0_Hermitian))
# bench_bunch_kaufman_rook_inplace_LAPACK = @benchmark bunchkaufman!(K, rook=true) setup=(K=copy($K_0_Hermitian))

println("Performance de bunch_kaufman:")
println(" - Allocations : ", mean(bench_bunch_kaufman).allocs)
println(" - Temps d'exécution : ", mean(bench_bunch_kaufman).time)

println("Performance de bunch_kaufman!:")
println(" - Allocations : ", mean(bench_bunch_kaufman_inplace).allocs)
println(" - Temps d'exécution : ", mean(bench_bunch_kaufman_inplace).time)

println("Performance de bunch_kaufman_rook:")
println(" - Allocations : ", mean(bench_bunch_kaufman_rook).allocs)
println(" - Temps d'exécution : ", mean(bench_bunch_kaufman_rook).time)

println("Performance de bunch_kaufman_rook!:")
println(" - Allocations : ", mean(bench_bunch_kaufman_rook_inplace).allocs)
println(" - Temps d'exécution : ", mean(bench_bunch_kaufman_rook_inplace).time)

println("Performance de bunchkaufman LAPACK:")
println(" - Allocations : ", mean(bench_bunch_kaufman_LAPACK).allocs)
println(" - Temps d'exécution : ", mean(bench_bunch_kaufman_LAPACK).time)

println("Performance de bunchkaufman! LAPACK:")
println(" - Allocations : ", mean(bench_bunch_kaufman_inplace_LAPACK).allocs)
println(" - Temps d'exécution : ", mean(bench_bunch_kaufman_inplace_LAPACK).time)

# println("Performance de bunchkaufman_rook LAPACK:")
# println(" - Allocations : ", mean(bench_bunch_kaufman_rook_LAPACK).allocs)
# println(" - Temps d'exécution : ", mean(bench_bunch_kaufman_rook_LAPACK).time)

# println("Performance de bunchkaufman_rook! LAPACK:")
# println(" - Allocations : ", mean(bench_bunch_kaufman_rook_inplace_LAPACK).allocs)
# println(" - Temps d'exécution : ", mean(bench_bunch_kaufman_rook_inplace_LAPACK).time)

Performance de bunch_kaufman:
 - Allocations : 18
 - Temps d'exécution : 9.795328172888016e6
Performance de bunch_kaufman!:
 - Allocations : 10
 - Temps d'exécution : 9.47149986907021e6
Performance de bunch_kaufman_rook:
 - Allocations : 22
 - Temps d'exécution : 8.839190322123894e6
Performance de bunch_kaufman_rook!:
 - Allocations : 14
 - Temps d'exécution : 9.425521086792452e6
Performance de bunchkaufman LAPACK:
 - Allocations : 71
 - Temps d'exécution : 6.577020864900663e6
Performance de bunchkaufman! LAPACK:
 - Allocations : 65
 - Temps d'exécution : 5.699916600686499e6


### Résolution de systèmes linéaires

In [125]:
import Base: \

function \(F::LBL{T}, b::AbstractVector{T}) where T
    L, B, vec_P = F.L, F.B, F.vec_P
    n = size(vec_P, 1)

    # Permuter b : b_perm = P * b
    b_perm = b[vec_P]

    # Résoudre L * y = b_perm <=> y = L \ b_perm
    y = similar(b)
    for i in 1:n
        y[i] = b_perm[i]
        for j in 1:i-1
            y[i] -= L[i, j]*y[j]
        end
    end

    # Résoudre B * z = y <=> z = B \ y
    z = similar(b)
    i = 1
    while i < n-1
        if B[i+1, i] == 0
            z[i] = y[i]/B[i, i]

            i += 1
        else
            B_11, B_22, B_21 = B[i, i], B[i+1, i+1], B[i+1, i]
            B_12 = conj(B_21)
            det = (B_11*B_22 - B_21*B_12)
            inv_det = conj(det)/abs(det)^2
            z[i] = inv_det*(B_22*y[i] - B_12*y[i+1])
            z[i+1] = inv_det*(B_11*y[i+1] - B_21*y[i])

            i += 2
        end
    end
    if B[n, n-1] == 0
        z[n-1] = y[n-1]/B[n-1, n-1]
        z[n] = y[n]/B[n, n]
    else
        B_11, B_22, B_21 = B[n-1, n-1], B[n, n], B[n, n-1]
        B_12 = conj(B_21)
        det = (B_11*B_22 - B_21*B_12)
        inv_det = conj(det)/abs(det)^2
        z[n-1] = inv_det*(B_22*y[n-1] - B_12*y[n])
        z[n] = inv_det*(B_11*y[n] - B_21*y[n-1])
    end

    # Résoudre L'* x_perm = z <=> x_perm = L' \ z
    x_perm = similar(b)
    for i in n:-1:1
        x_perm[i] = z[i]
        for j in n:-1:i+1
            x_perm[i] -= conj(L[j, i])*x_perm[j]
        end
    end

    # Permuter x_perm : x = P' * x_perm
    x = similar(b)
    x[vec_P] = x_perm  # Inverse de P est P' car P est une permutation

    return x
end

\ (generic function with 160 methods)

#### Comparaison sur des problèmes aux moindres carrés aléatoires

In [126]:
N = 100
m, n = 50, 25
list_error_bunch_kaufman = zeros(N)
list_error_bunch_kaufman_inplace = zeros(N)
list_error_bunch_kaufman_rook = zeros(N)
list_error_bunch_kaufman_rook_inplace = zeros(N)
list_error_bunch_kaufman_LAPACK = zeros(N)
list_error_bunch_kaufman_inplace_LAPACK = zeros(N)
for i in 1:N
    A = randn(BigFloat, m, n) + im * randn(BigFloat, m, n)
    b = randn(BigFloat, m) + im * randn(BigFloat, m)
    x_qr = qr(A) \ b

    T = eltype(A)
    K_0 = LowerTriangular([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    K_0_Hermitian = Hermitian([sparse(Matrix{T}(I, m, m)) A; A' spzeros(n,n)])
    rhs_0 = [b; zeros(T, n)]
    rhs = copy(rhs_0)

    K = copy(K_0)
    F = bunch_kaufman(K)
    sol = F \ rhs
    x = sol[m+1:end]
    list_error_bunch_kaufman[i] = norm(x - x_qr)

    K = copy(K_0)
    vec_P, vec_2by2 = bunch_kaufman!(K)
    L, B = extract_L_and_B_from_LBL_inplace(K, vec_2by2)
    F = LBL(L, B, vec_P)
    sol = F \ rhs
    x = sol[m+1:end]
    list_error_bunch_kaufman_inplace[i] = norm(x - x_qr)

    K = copy(K_0)
    F = bunch_kaufman_rook(K)
    sol = F \ rhs
    x = sol[m+1:end]
    list_error_bunch_kaufman_rook[i] = norm(x - x_qr)

    K = copy(K_0)
    vec_P, vec_2by2 = bunch_kaufman_rook!(K)
    L, B = extract_L_and_B_from_LBL_inplace(K, vec_2by2)
    F = LBL(L, B, vec_P)
    sol = F \ rhs
    x = sol[m+1:end]
    list_error_bunch_kaufman_rook_inplace[i] = norm(x - x_qr)

    K = copy(K_0_Hermitian)
    F_LAPACK = bunchkaufman(K)
    sol = F_LAPACK \ rhs
    x_LAPACK = sol[m+1:end]
    list_error_bunch_kaufman_LAPACK[i] = norm(x_LAPACK - x_qr)

    K = copy(K_0_Hermitian)
    F_LAPACK = bunchkaufman!(K)
    sol = F_LAPACK \ rhs
    x_LAPACK = sol[m+1:end]
    list_error_bunch_kaufman_inplace_LAPACK[i] = norm(x_LAPACK - x_qr)
end

println("Average error on solution:")
println(" - bunch_kaufman : ", mean(list_error_bunch_kaufman))
println(" - bunch_kaufman! : ", mean(list_error_bunch_kaufman_inplace))
println(" - bunch_kaufman_rook : ", mean(list_error_bunch_kaufman_rook))
println(" - bunch_kaufman_rook! : ", mean(list_error_bunch_kaufman_rook_inplace))
println(" - bunch_kaufman de LAPACK : ", mean(list_error_bunch_kaufman_LAPACK))
println(" - bunch_kaufman! de LAPACK : ", mean(list_error_bunch_kaufman_LAPACK))

Average error on solution:
 - bunch_kaufman : 1.489835887139084e-76
 - bunch_kaufman! : 1.489835887139084e-76
 - bunch_kaufman_rook : 2.296976563833339e-74
 - bunch_kaufman_rook! : 2.296976563833339e-74
 - bunch_kaufman de LAPACK : 2.822133712442099e-76
 - bunch_kaufman! de LAPACK : 2.822133712442099e-76


#### Comparaison sur des problèmes aux moindres carrés de Matrix Market (Suite Sparse Collection)

Les résolutions sur système linéaire offertes par l'implémentation de bunchkaufman avec LinearAlgebra semblent avoir une restriction de matrice inversible qui empêche leur exécution sur des systèmes typiques de Matrix Market (Suite Sparse Collection).

In [127]:
function get_mm(matrix_name)
    ssmc = ssmc_db()
    pb = ssmc_matrices(ssmc, "", matrix_name)
    fetch_ssmc(pb, format="MM")
    pb_path = fetch_ssmc(pb, format="MM")
    path_mtx = pb_path[1]
    A = MatrixMarket.mmread(joinpath(path_mtx, matrix_name * ".mtx"));
    m, n = size(A)
    b = randn(m)
    # b = MatrixMarket.mmread(joinpath(path_mtx, matrix_name * "_b.mtx"))[:];

    K = [sparse(1.0I,m,m) A; A' spzeros(n,n)]
    rhs = [b; zeros(n)]    
    return A, b, K, rhs
end

get_mm (generic function with 1 method)

In [128]:
list_names = ["ch5-5-b1", "ch4-4-b1", "ch4-4-b2", "n3c5-b2"]
N = size(list_names, 1)

list_error_bunch_kaufman = zeros(N)
list_error_bunch_kaufman_inplace = zeros(N)
list_error_bunch_kaufman_rook = zeros(N)
list_error_bunch_kaufman_rook_inplace = zeros(N)
list_error_bunch_kaufman_LAPACK = zeros(N)
list_error_bunch_kaufman_inplace_LAPACK = zeros(N)
for i in 1:N
    name = list_names[i]
    A, b, K_0, rhs_0 = get_mm(name)
    m, n = size(A)
    x_qr = qr(A) \ b

    K_0 = LowerTriangular(K_0)
    K_0_Hermitian = Hermitian(K_0)
    rhs = copy(rhs_0)

    K = copy(K_0)
    F = bunch_kaufman(K)
    sol = F \ rhs
    x = sol[m+1:end]
    list_error_bunch_kaufman[i] = norm(A*x - b) - norm(A*x_qr - b)

    K = copy(K_0)
    vec_P, vec_2by2 = bunch_kaufman!(K)
    L, B = extract_L_and_B_from_LBL_inplace(K, vec_2by2)
    F = LBL(L, B, vec_P)
    sol = F \ rhs
    x = sol[m+1:end]
    list_error_bunch_kaufman_inplace[i] = norm(A*x - b) - norm(A*x_qr - b)

    K = copy(K_0)
    F = bunch_kaufman_rook(K)
    sol = F \ rhs
    x = sol[m+1:end]
    list_error_bunch_kaufman_rook[i] = norm(A*x - b) - norm(A*x_qr - b)

    K = copy(K_0)
    vec_P, vec_2by2 = bunch_kaufman_rook!(K)
    L, B = extract_L_and_B_from_LBL_inplace(K, vec_2by2)
    F = LBL(L, B, vec_P)
    sol = F \ rhs
    x = sol[m+1:end]
    list_error_bunch_kaufman_rook_inplace[i] = norm(A*x - b) - norm(A*x_qr - b)

    # K = copy(K_0_Hermitian)
    # F_LAPACK = bunchkaufman(K)
    # sol = F_LAPACK \ rhs
    # x_LAPACK = sol[m+1:end]
    # list_error_bunch_kaufman_LAPACK[i] = norm(x_LAPACK - x_qr)

    # K = copy(K_0_Hermitian)
    # F_LAPACK = bunchkaufman!(K)
    # sol = F_LAPACK \ rhs
    # x_LAPACK = sol[m+1:end]
    # list_error_bunch_kaufman_inplace_LAPACK[i] = norm(x_LAPACK - x_qr)

    println("Error on residual for " * name * " :")
    println(" - bunch_kaufman : ", list_error_bunch_kaufman[i])
    println(" - bunch_kaufman! : ", list_error_bunch_kaufman_inplace[i])
    println(" - bunch_kaufman_rook : ", list_error_bunch_kaufman_rook[i])
    println(" - bunch_kaufman_rook! : ", list_error_bunch_kaufman_rook_inplace[i])
    # println(" - bunch_kaufman de LAPACK : ", list_error_bunch_kaufman_LAPACK[i])
    # println(" - bunch_kaufman! de LAPACK : ", list_error_bunch_kaufman_LAPACK[i])
end

println("Average error on residual:")
println(" - bunch_kaufman : ", mean(list_error_bunch_kaufman))
println(" - bunch_kaufman! : ", mean(list_error_bunch_kaufman_inplace))
println(" - bunch_kaufman_rook : ", mean(list_error_bunch_kaufman_rook))
println(" - bunch_kaufman_rook! : ", mean(list_error_bunch_kaufman_rook_inplace))
# println(" - bunch_kaufman de LAPACK : ", mean(list_error_bunch_kaufman_LAPACK))
# println(" - bunch_kaufman! de LAPACK : ", mean(list_error_bunch_kaufman_LAPACK))

Error on residual for ch5-5-b1 :
 - bunch_kaufman : 0.0008936207727661127
 - bunch_kaufman! : 0.0008936207727661127
 - bunch_kaufman_rook : 0.0008936207727661127
 - bunch_kaufman_rook! : 0.0008936207727661127
Error on residual for ch4-4-b1 :
 - bunch_kaufman : 1.4559844559336454e-5
 - bunch_kaufman! : 1.4559844559336454e-5
 - bunch_kaufman_rook : 1.4559844559336454e-5
 - bunch_kaufman_rook! : 1.4559844559336454e-5
Error on residual for ch4-4-b2 :
 - bunch_kaufman : 0.0
 - bunch_kaufman! : 0.0
 - bunch_kaufman_rook : 0.0
 - bunch_kaufman_rook! : 0.0
Error on residual for n3c5-b2 :
 - bunch_kaufman : -1.7763568394002505e-15
 - bunch_kaufman! : -1.7763568394002505e-15
 - bunch_kaufman_rook : -1.7763568394002505e-15
 - bunch_kaufman_rook! : -1.7763568394002505e-15
Average error on residual:
 - bunch_kaufman : 0.0002270451543309182
 - bunch_kaufman! : 0.0002270451543309182
 - bunch_kaufman_rook : 0.0002270451543309182
 - bunch_kaufman_rook! : 0.0002270451543309182
